# Pulsar Candidate Selection v2 â€” 1D-CNN replication (HTRU2)

**Goal.** Replicate the published CNN-on-HTRU2-tabular experiment as precisely as possible, on the *identical* splits as v1 (SEED=42), and compare paper-style: literature vs RF vs MLP vs CNN.

**Replication target.** UCSD ECE228 Report6 (2019): LR/RF/CNN/MLP on the HTRU2 8-dim vectors; CNN search = 1â€“4 conv layers, filter sizes (6,2,1)â†’(32,4,1), max-pool on/off, tanh/ReLU, SGD. Reference anchor: Lyon et al. 2016 GH-VFDT (acc 0.978, prec 0.899, recall 0.829).

**Hard boundary.** Image-based systems (Lin et al. 2020 MICNN on 4 diagnostic plots; PICS/CoAtNet; FAST multimodal work) need raw candidate *plot pixels*, which HTRU2 does not contain (8 statistics only). They are cited, not claimed â€” replicating them from this data would be fabrication.


## 1. Replication card â€” what is copied exactly, what deviates

| Item | Literature | This replication |
|---|---|---|
| Input | 8-dim HTRU2 vector | Same, standardised; viewed as length-8 1-channel sequence |
| Splits | Report6: unspecified random split | v1 splits reused exactly (stratified 60/20/20, SEED=42) â€” stricter and comparable |
| Imbalance | Report6: "statistical balancing" (unspecified) | Explicit class-weighted cross-entropy (weights = inverse train frequency); deviation logged, not hidden |
| CNN space | 1â€“4 conv layers, filters (6,2,1)â†’(32,4,1), pool on/off, tanh/ReLU, SGD | Grid: blocks {1,2} Ã— channels {8,16} Ã— kernel {2,3} Ã— pool {on,off} Ã— act {relu,tanh} = 32 configs, SGD lr=0.01 momentum 0.9, val PR-AUC selection |
| MLP | Report6: MLP among 4 models, config unspecified | sklearn MLPClassifier (100,), early stopping â€” baseline only |
| Metrics | Report6 Table 3: acc, neg/pos precision, neg/pos recall | Same five + v1's ROC-AUC / PR-AUC / Brier + same 10:1 cost threshold |


In [ ]:
SEED = 42
import random, sys
import numpy as np, pandas as pd
random.seed(SEED); np.random.seed(SEED)
import torch
torch.manual_seed(SEED)
torch.set_num_threads(4)
import sklearn, matplotlib
print(f"python {sys.version.split()[0]} | torch {torch.__version__} (cpu={not torch.cuda.is_available()}) | sklearn {sklearn.__version__}")

In [ ]:
from pathlib import Path
import urllib.request, zipfile, io
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
DATA = Path("data") / "HTRU_2.csv"
if not DATA.exists():
    DATA.parent.mkdir(exist_ok=True)
    z = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(
        "https://archive.ics.uci.edu/static/public/372/htru2.zip", timeout=120).read()))
    DATA.write_bytes(z.read("HTRU_2.csv"))
COLS = ["profile_mean","profile_stdev","profile_kurtosis","profile_skewness",
        "dm_mean","dm_stdev","dm_kurtosis","dm_skewness","target"]
df = pd.read_csv(DATA, header=None, names=COLS)
feats = COLS[:8]
X, y = df[feats].values, df["target"].values
X_tr, X_tmp, y_tr, y_tmp = train_test_split(X, y, test_size=0.40, stratify=y, random_state=SEED)
X_va, X_te, y_va, y_te = train_test_split(X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=SEED)
sc = StandardScaler().fit(X_tr)
Z_tr, Z_va, Z_te = sc.transform(X_tr), sc.transform(X_va), sc.transform(X_te)
print(f"identical v1 splits: train={len(y_tr)} val={len(y_va)} test={len(y_te)} | test pos_rate={y_te.mean():.4f}")
w = len(y_tr) / (2 * np.bincount(y_tr))
print(f"CE class weights (neg,pos) = ({w[0]:.3f},{w[1]:.3f})")

## 2. MLP baseline (sklearn, fast) â€” same splits, same metrics

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, roc_auc_score, average_precision_score,
    brier_score_loss, precision_score, recall_score)
def report5(name, yt, p, yhat=None):
    yh = (p >= 0.5).astype(int) if yhat is None else yhat
    tn = ((yh==0)&(yt==0)).sum(); fp = ((yh==1)&(yt==0)).sum()
    fn = ((yh==0)&(yt==1)).sum(); tp = ((yh==1)&(yt==1)).sum()
    r = dict(acc=(tp+tn)/len(yt), roc=roc_auc_score(yt,p), pr=average_precision_score(yt,p),
        brier=brier_score_loss(yt,p), negprec=tn/max(tn+fn,1), posprec=tp/max(tp+fp,1),
        negrec=tn/max(tn+fp,1), posrec=tp/max(tp+fn,1))
    print(f"{name}: acc={r['acc']:.4f} roc={r['roc']:.4f} prauc={r['pr']:.4f} brier={r['brier']:.4f} "
          f"negprec={r['negprec']:.4f} posprec={r['posprec']:.4f} negrec={r['negrec']:.4f} posrec={r['posrec']:.4f}")
    return r
mlp = MLPClassifier((100,), early_stopping=True, validation_fraction=0.2, n_iter_no_change=10,
    random_state=SEED).fit(Z_tr, y_tr)
mlp_r = report5("mlp", y_te, mlp.predict_proba(Z_te)[:,1])

## 3. 1D-CNN grid over the published search space (SGD, val PR-AUC selection)

In [ ]:
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
class CNN1d(nn.Module):
    def __init__(self, blocks=1, ch=8, k=3, pool=True, act="relu"):
        super().__init__()
        f = nn.ReLU if act=="relu" else nn.Tanh
        layers, c_in = [], 1
        for _ in range(blocks):
            layers += [nn.Conv1d(c_in, ch, k, padding=k//2), f()]
            c_in = ch
            if pool: layers += [nn.MaxPool1d(2)]
        self.conv = nn.Sequential(*layers)
        with torch.no_grad():
            n_flat = self.conv(torch.zeros(1, 1, 8)).numel()
        self.fc = nn.Linear(n_flat, 2)
    def forward(self, x): return self.fc(self.conv(x).flatten(1))
def loader(Z, y, bs=256, shuffle=False, seed=SEED):
    ds = TensorDataset(torch.tensor(Z, dtype=torch.float32).unsqueeze(1), torch.tensor(y))
    g = torch.Generator().manual_seed(seed) if shuffle else None
    return DataLoader(ds, batch_size=bs, shuffle=shuffle, num_workers=0, generator=g)
import json as _json
# Compute step lives in cnn_grid.py (32 configs, SGD, early-stop on val CE, val PR-AUC selection).
# The notebook only reports it — re-running the grid here would take ~10 min per execution.
G = _json.load(open("cnn_grid_results.json"))
print(f"grid: {G['grid']} configs, seed={G['seed']}")
print("top-5 by val PR-AUC:")
for r in G["rows"][:5]:
    print(f"  {r['val_prauc']:.4f} eps={r['epochs']} {r['cfg']}")
best_cfg, hist = G["winner"]["cfg"], G["hist"]
print("\nselected:", best_cfg, f"val-PRAUC={G['winner']['val_prauc']:.4f}")

## 4. Finalist â€” test metrics, cost threshold, reliability, training curves

In [ ]:
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve
cnn = CNN1d(**best_cfg)
cnn.load_state_dict(torch.load("cnn_best.pt", weights_only=True)["state"])
cnn.eval()
with torch.no_grad():
    p_te = torch.cat([cnn(xb) for xb,_ in loader(Z_te, y_te)]).softmax(1)[:,1].numpy()
    p_va = torch.cat([cnn(xb) for xb,_ in loader(Z_va, y_va)]).softmax(1)[:,1].numpy()
print("finalist:", best_cfg)
cnn_r = report5("cnn", y_te, p_te)
C_FN, C_FP, T = 10, 1, None
ths = np.linspace(0.05, 0.95, 19)
T = float(ths[int(np.argmin([C_FN*((p_va<t)&(y_va==1)).sum() + C_FP*((p_va>=t)&(y_va==0)).sum() for t in ths]))])
yt = (p_te >= T).astype(int)
print(f"cost threshold T={T:.2f} -> prec={precision_score(y_te,yt):.3f} recall={recall_score(y_te,yt):.3f}")
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
ax[0].plot(hist["loss"], label="train CE loss"); ax[0].set_title("CNN training loss"); ax[0].legend()
ax[1].plot(hist["val_prauc"], label="val PR-AUC", color="green"); ax[1].set_title("CNN validation PR-AUC"); ax[1].legend()
plt.tight_layout(); plt.savefig("docs/assets/cnn_train.png", bbox_inches="tight"); plt.show()
fig, ax = plt.subplots(figsize=(4.6, 4))
ax.plot([0,1],[0,1], ls="--", label="ideal")
ft, mp = calibration_curve(y_te, p_te, n_bins=10, strategy="quantile")
ax.plot(mp, ft, marker="o", ms=4, label="cnn (uncalibrated)")
ax.set_title("CNN reliability (test)"); ax.set_xlabel("mean predicted"); ax.set_ylabel("fraction positive")
ax.legend(); plt.tight_layout(); plt.savefig("docs/assets/cnn_reliability.png", bbox_inches="tight"); plt.show()

## 5. Paper-style comparison + verdict

Fill the `LITERATURE` row from the cited papers (Lyon 2016 GH-VFDT: acc 0.978 / prec 0.899 / recall 0.829; Report6 Table 3 CNN: acc 0.9837 / pos-rec 0.9746 / pos-prec 0.9926). Our rows come from this notebook on identical splits. Verdict and deviations are written after the run â€” the table below prints ours; the literature column is quoted, not recomputed.


In [ ]:
from sklearn.ensemble import RandomForestClassifier
# v1 RF retrained verbatim (best params, same splits) so every row below is measured, none quoted.
rf = RandomForestClassifier(n_estimators=400, max_depth=20, min_samples_leaf=2,
    class_weight="balanced_subsample", random_state=SEED, n_jobs=-1).fit(X_tr, y_tr)
rf_r = report5("rf-v1", y_te, rf.predict_proba(X_te)[:,1])
print(f"{'model':<12}{'acc':<8}{'roc':<8}{'prauc':<8}{'posprec':<9}{'posrec':<8}{'negrec':<8}")
for name, r in [("rf-v1", rf_r), ("mlp", mlp_r), ("cnn", cnn_r)]:
    print(f"{name:<12}{r['acc']:<8.4f}{r['roc']:<8.4f}{r['pr']:<8.4f}{r['posprec']:<9.4f}{r['posrec']:<8.4f}{r['negrec']:<8.4f}")
print("\nliterature (quoted, not recomputed): Lyon2016 GH-VFDT acc=0.978 prec=0.899 recall=0.829 | "
      "Report6 CNN acc=0.9837 posprec=0.9926 posrec=0.9746")

**Deviations log (viva defence).** (1) Splits: ours are fixed/stratified, theirs unspecified â€” ours is the stricter claim. (2) Balancing: explicit class weights vs their undescribed statistics. (3) Filter-size notation `(6,2,1)â†’(32,4,1)` is ambiguous in the source; we searched channels {8,16} Ã— kernel {2,3} and report the val-selected config instead of guessing theirs. (4) CNN is uncalibrated (reliability shown raw); temperature scaling is the named next step, not a silent omission.

**Repro checklist.** SEED=42 (python/numpy/torch, per-config reseed) Â· identical v1 splits Â· SGD as published Â· val PR-AUC selection Â· Report6's five columns + v1's ROC/PR/Brier Â· cost threshold reused Â· assets written to `docs/assets/` by Â§4.
